# physics_solution_class_feature_selection_v2.1

tools/physics_solution 기준으로 motion/onset/geometry feature를 추출하고, stable/unstable 상관관계를 분석합니다.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score


ROOT = Path.cwd().resolve()
if not (ROOT / "tools" / "physics_solution").exists() and (ROOT.parent / "tools" / "physics_solution").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "tools" / "physics_solution"))
from full_physics_solution import MotionExtractionConfig, extract_motion_targets
from geometry_reasoning import GeometryReasoningConfig, extract_geometry_features


DATA_ROOT = ROOT / "data"
OUT_DIR = ROOT / "outputs" / "physics_feature_analysis_v2_1"
MOTION_CSV = DATA_ROOT / "motion_targets.csv"
FORCE_REEXTRACT_MOTION = False
INCLUDE_DEV_IN_ANALYSIS = True
MAX_SAMPLES_PER_SPLIT: int | None = None
LABEL_MAP = {"stable": 0, "unstable": 1}


OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR:", OUT_DIR)


def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, encoding="utf-8-sig")


def read_rgb(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def ensure_motion_targets(data_root: Path, motion_csv: Path, force_refresh: bool) -> pd.DataFrame:
    if force_refresh or (not motion_csv.exists()):
        print("extracting motion targets ->", motion_csv)
        motion_df = extract_motion_targets(data_root, motion_csv, MotionExtractionConfig())
    else:
        motion_df = read_csv(motion_csv)
    return motion_df


def build_sample_meta(data_root: Path, include_dev: bool, max_samples_per_split: int | None) -> pd.DataFrame:
    rows = []
    for split in ["train", "dev"] if include_dev else ["train"]:
        df = read_csv(data_root / f"{split}.csv")
        if max_samples_per_split is not None:
            df = df.head(max_samples_per_split).copy()
        for row in df.itertuples(index=False):
            rows.append({"split": split, "sample_id": row.id, "label": row.label})
    return pd.DataFrame(rows)


def build_geometry_df(data_root: Path, sample_meta: pd.DataFrame) -> pd.DataFrame:
    cfg = GeometryReasoningConfig()
    rows = []
    for row in sample_meta.itertuples(index=False):
        front_path = data_root / row.split / row.sample_id / "front.png"
        top_path = data_root / row.split / row.sample_id / "top.png"
        if not front_path.exists() or not top_path.exists():
            continue
        front_rgb = read_rgb(front_path)
        top_rgb = read_rgb(top_path)
        feat = extract_geometry_features(front_rgb, top_rgb, cfg)
        feat["collapse_margin_proxy"] = float(
            np.clip(
                0.5
                + 0.35
                * (
                    1.20 * feat["top_support_width_frac"]
                    + 0.90 * feat["front_base_width_frac"]
                    + 0.50 * feat["top_fill_ratio"]
                    - 0.75 * abs(feat["top_centroid_dx"])
                    - 0.55 * abs(feat["front_tilt"])
                    - 0.20 * feat["front_slenderness"]
                    - 0.25 * feat["front_top_heaviness"]
                ),
                0.0,
                1.0,
            )
        )
        feat.update({"sample_id": row.sample_id, "split": row.split, "label": row.label})
        rows.append(feat)
    return pd.DataFrame(rows)


def select_motion_cols(motion_df: pd.DataFrame) -> pd.DataFrame:
    keep_cols = [
        "id",
        "max_diff_first",
        "mean_diff_first",
        "max_diff_prev",
        "mean_diff_prev",
        "first_move_thr2",
        "first_move_thr5",
        "first_move_thr10",
        "severity_bucket",
        "onset_bucket",
        "soft_target",
    ]
    keep_cols = [c for c in keep_cols if c in motion_df.columns]
    out = motion_df[keep_cols].copy()
    out = out.rename(columns={"id": "sample_id"})
    return out


def build_analysis_df(sample_meta: pd.DataFrame, geometry_df: pd.DataFrame, motion_df: pd.DataFrame) -> pd.DataFrame:
    merged = sample_meta.merge(
        geometry_df.drop(columns=["split", "label"], errors="ignore"),
        on="sample_id",
        how="left",
    )
    merged = merged.merge(motion_df, on="sample_id", how="left")
    merged["target"] = merged["label"].map(LABEL_MAP)
    return merged


def _safe_auc(y_true: np.ndarray, score: np.ndarray) -> float:
    valid = np.isfinite(score) & np.isfinite(y_true)
    if valid.sum() < 4:
        return float("nan")
    y = y_true[valid].astype(int)
    s = score[valid]
    if len(np.unique(y)) < 2 or np.nanstd(s) == 0:
        return float("nan")
    auc = float(roc_auc_score(y, s))
    return max(auc, 1.0 - auc)


def _cohen_d(stable_vals: np.ndarray, unstable_vals: np.ndarray) -> float:
    a = stable_vals[np.isfinite(stable_vals)]
    b = unstable_vals[np.isfinite(unstable_vals)]
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    va = np.var(a, ddof=1)
    vb = np.var(b, ddof=1)
    pooled = np.sqrt(((len(a) - 1) * va + (len(b) - 1) * vb) / max(len(a) + len(b) - 2, 1))
    if pooled == 0:
        return float("nan")
    return float((np.mean(b) - np.mean(a)) / pooled)


def compute_feature_correlation_tables(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    excluded = {"sample_id", "split", "label", "target"}
    feature_cols = sorted([c for c in df.columns if c not in excluded and df[c].notna().any()])

    corr_rows = []
    stats_rows = []

    work = df.copy()
    work["target"] = work["target"].astype(float)

    stable_mask = work["target"].eq(0)
    unstable_mask = work["target"].eq(1)
    y = work["target"].to_numpy(dtype=np.float64)

    for col in feature_cols:
        x = pd.to_numeric(work[col], errors="coerce").to_numpy(dtype=np.float64)
        tmp = pd.DataFrame({"x": x, "y": y}).dropna()
        if len(tmp) >= 4 and tmp["x"].nunique() > 1 and tmp["y"].nunique() > 1:
            pearson = float(tmp["x"].corr(tmp["y"], method="pearson"))
            spearman = float(tmp["x"].corr(tmp["y"], method="spearman"))
        else:
            pearson = float("nan")
            spearman = float("nan")

        auc = _safe_auc(y, x)
        stable_vals = pd.to_numeric(work.loc[stable_mask, col], errors="coerce").to_numpy(dtype=np.float64)
        unstable_vals = pd.to_numeric(work.loc[unstable_mask, col], errors="coerce").to_numpy(dtype=np.float64)
        d = _cohen_d(stable_vals, unstable_vals)

        corr_rows.append(
            {
                "feature": col,
                "pearson_corr_with_target": pearson,
                "spearman_corr_with_target": spearman,
                "abs_pearson_corr": abs(pearson) if np.isfinite(pearson) else np.nan,
                "abs_spearman_corr": abs(spearman) if np.isfinite(spearman) else np.nan,
                "single_feature_auc_abs": auc,
                "cohen_d_unstable_minus_stable": d,
                "abs_cohen_d": abs(d) if np.isfinite(d) else np.nan,
                "n_valid": int(np.isfinite(x).sum()),
            }
        )

        stats_rows.append(
            {
                "feature": col,
                "stable_n": int(np.isfinite(stable_vals).sum()),
                "stable_mean": float(np.nanmean(stable_vals)) if np.isfinite(stable_vals).any() else np.nan,
                "stable_std": float(np.nanstd(stable_vals)) if np.isfinite(stable_vals).any() else np.nan,
                "stable_median": float(np.nanmedian(stable_vals)) if np.isfinite(stable_vals).any() else np.nan,
                "unstable_n": int(np.isfinite(unstable_vals).sum()),
                "unstable_mean": float(np.nanmean(unstable_vals)) if np.isfinite(unstable_vals).any() else np.nan,
                "unstable_std": float(np.nanstd(unstable_vals)) if np.isfinite(unstable_vals).any() else np.nan,
                "unstable_median": float(np.nanmedian(unstable_vals)) if np.isfinite(unstable_vals).any() else np.nan,
                "mean_gap_unstable_minus_stable": (
                    float(np.nanmean(unstable_vals) - np.nanmean(stable_vals))
                    if np.isfinite(stable_vals).any() and np.isfinite(unstable_vals).any()
                    else np.nan
                ),
            }
        )

    corr_df = pd.DataFrame(corr_rows).sort_values(
        ["single_feature_auc_abs", "abs_pearson_corr", "abs_cohen_d"],
        ascending=[False, False, False],
    )
    stats_df = pd.DataFrame(stats_rows)
    return corr_df.reset_index(drop=True), stats_df.reset_index(drop=True)


def build_bucket_tables(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    out = {}
    for col in ["severity_bucket", "onset_bucket"]:
        if col not in df.columns:
            continue
        tab = (
            df.groupby([col, "label"], dropna=False)
            .size()
            .rename("count")
            .reset_index()
            .pivot(index=col, columns="label", values="count")
            .fillna(0)
        )
        tab["total"] = tab.sum(axis=1)
        for label in ["stable", "unstable"]:
            if label not in tab.columns:
                tab[label] = 0
        tab["unstable_ratio"] = tab["unstable"] / tab["total"].clip(lower=1)
        out[col] = tab.reset_index()
    return out


sample_meta = build_sample_meta(DATA_ROOT, INCLUDE_DEV_IN_ANALYSIS, MAX_SAMPLES_PER_SPLIT)
motion_raw_df = ensure_motion_targets(DATA_ROOT, MOTION_CSV, FORCE_REEXTRACT_MOTION)
motion_df = select_motion_cols(motion_raw_df)
geometry_df = build_geometry_df(DATA_ROOT, sample_meta)
analysis_df = build_analysis_df(sample_meta, geometry_df, motion_df)

corr_df, stats_df = compute_feature_correlation_tables(analysis_df)
bucket_tables = build_bucket_tables(analysis_df)

analysis_df.to_csv(OUT_DIR / "physics_features_train_dev_v2.1.csv", index=False, encoding="utf-8-sig")
corr_df.to_csv(OUT_DIR / "physics_feature_target_correlation_v2.1.csv", index=False, encoding="utf-8-sig")
stats_df.to_csv(OUT_DIR / "physics_feature_stable_unstable_stats_v2.1.csv", index=False, encoding="utf-8-sig")
for name, tab in bucket_tables.items():
    tab.to_csv(OUT_DIR / f"{name}_stable_unstable_table_v2.1.csv", index=False, encoding="utf-8-sig")

print("saved:", OUT_DIR)
print("samples:", len(analysis_df))
print("features:", len(corr_df))
print("\nTop correlated features:")
display(corr_df.head(25))
print("\nStable/Unstable descriptive stats (head):")
display(stats_df.head(25))
for name, tab in bucket_tables.items():
    print(f"\n{name} distribution:")
    display(tab)

